In [12]:
import json

file_path = "/content/drive/MyDrive/DL LAB S2/8/Dataset/train.json"
with open(file_path, "r") as f:
    train_json_content = json.load(f)

print("Content of train.json:")
print(json.dumps(train_json_content, indent=2))

Content of train.json:
[
  {
    "context": "Python is a high-level, general-purpose programming language created by Guido van Rossum.",
    "qas": [
      {
        "id": "00001",
        "is_impossible": false,
        "question": "Who created the Python programming language?",
        "answers": [
          {
            "text": "Guido van Rossum",
            "answer_start": 72
          }
        ]
      }
    ]
  },
  {
    "context": "The first generation of computers, developed between 1940 and 1956, consisted of vacuum tubes, magnetic drums, and punched cards.",
    "qas": [
      {
        "id": "00002",
        "is_impossible": false,
        "question": "When were the first generation of computers developed?",
        "answers": [
          {
            "text": "between 1940 and 1956",
            "answer_start": 45
          }
        ]
      },
      {
        "id": "00003",
        "is_impossible": false,
        "question": "What were the main components used in the fi

### Refactored QA System

In [13]:
# Imports
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch
import json
import re

In [14]:
# Helper function to load data
def load_data(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

In [25]:
def load_and_process_test_data(file_path):
    data = load_data(file_path)
    contexts = []
    qas = []
    for item in data:
        contexts.append(item["context"])
        for qa_item in item["qas"]:
            qas.append({
                "id": qa_item["id"],
                "question": qa_item["question"],
                "context": item["context"],
                "answers": qa_item["answers"],
                "is_impossible": qa_item["is_impossible"]
            })
    return contexts, qas

In [15]:
# Load training data and extract contexts
train_data = load_data("/content/drive/MyDrive/DL LAB S2/8/Dataset/train.json")
contexts = [item["context"] for item in train_data]
print(f"\n Loaded {len(contexts)} contexts")


 Loaded 5 contexts


In [16]:
# Load pre-trained model and tokenizer
print("\n Loading model...")
model_name = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
print(" Model loaded!")


 Loading model...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

 Model loaded!


In [17]:
# Function to find the best context for a given question
def find_best_context(question):
    question_words = re.findall(r'\w+', question.lower())
    best_context = ""
    max_match = 0
    for context in contexts:
        match = sum(1 for word in question_words if word in context.lower())
        if match > max_match:
            max_match = match
            best_context = context
    if max_match < 2:
        return None
    return best_context

In [18]:
# Function to answer a question given a context
def answer_question(question, context):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = model(**inputs)
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits) + 1
    answer = tokenizer.decode(inputs["input_ids"][0][start:end])
    return answer.strip()

In [19]:
# Interactive Q&A loop
while True:
    question = input("\n❓ Ask a question (type 'exit' to quit): ")
    if question.lower() == "exit":
        print(" Exiting...")
        break
    context = find_best_context(question)
    if context is None:
        print(" No relevant context found")
        continue
    answer = answer_question(question, context)
    if answer:
        print("\n Answer:", answer)
    else:
        print("\n No answer found")


❓ Ask a question (type 'exit' to quit): When was the World Wide Web invented?

 Answer: 1989

❓ Ask a question (type 'exit' to quit): Who created the Python programming language?

 Answer: Guido van Rossum

❓ Ask a question (type 'exit' to quit): exit
 Exiting...
